# Лабораторная работа №5. Дерево решений

Реализуем классификатор «дерево решений» с нуля и сравним его с
`DecisionTreeClassifier` из `sklearn`.

По заданию необходимо:

1. Реализовать функцию расчёта критерия Джини.
2. Реализовать расчёт прироста информации.
3. Реализовать критерии останова (не менее двух).
4. Реализовать функцию подсчёта метрики качества модели.
5. Проверить работу самописного дерева и сравнить результат с `sklearn`
   (результаты должны совпасть полностью).

Код написан так, что не привязан к конкретному датасету: достаточно указать
путь к CSV-файлу, а целевой столбец по умолчанию — последний.


## 0. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


## 1. Универсальная загрузка данных

Функция не зависит от конкретного датасета:

* целевой столбец — последний (либо задаётся именем в аргументе);
* категориальная целевая переменная кодируется числами;
* нечисловые признаки с большим числом уникальных значений считаются
  идентификаторами и удаляются, остальные кодируются one-hot.


In [ ]:
def load_dataset(path, target_column=None, max_categories=50):
    """
    Универсальная загрузка табличного датасета из CSV.

    Параметры
    ----------
    path : str
        путь к CSV-файлу.
    target_column : str, optional
        имя целевого столбца; если None — берётся последний столбец.
    max_categories : int
        нечисловые столбцы с числом уникальных значений больше этого
        считаются идентификаторами и удаляются; остальные кодируются one-hot.

    Возвращает
    ----------
    X : np.ndarray, форма (n_объектов, n_признаков)
    y : np.ndarray, форма (n_объектов,)
    feature_names : list[str]
    """
    df = pd.read_csv(path)

    if target_column is None:
        target_column = df.columns[-1]

    y = df[target_column]
    X = df.drop(columns=[target_column])

    # Целевую переменную приводим к числам, если она категориальная
    if not np.issubdtype(y.dtype, np.number):
        y = y.astype('category').cat.codes

    # Нечисловые признаки: идентификаторы удаляем, категории кодируем one-hot
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    to_drop = [c for c in cat_cols if X[c].nunique() > max_categories]
    to_encode = [c for c in cat_cols if c not in to_drop]

    X = X.drop(columns=to_drop)
    if to_encode:
        X = pd.get_dummies(X, columns=to_encode)

    if to_drop:
        print(f'Удалены столбцы-идентификаторы: {to_drop}')
    if to_encode:
        print(f'Закодированы one-hot: {to_encode}')

    return X.to_numpy(dtype=float), y.to_numpy(), X.columns.tolist()


# Достаточно указать путь к файлу — код не привязан к именам столбцов
DATA_PATH = 'healthy_meal_plans.csv'

X, y, feature_names = load_dataset(DATA_PATH)
print(f'Объектов: {X.shape[0]}, признаков: {X.shape[1]}')
print(f'Классы: {np.unique(y)}')


## 2. Классы узла и листа дерева

Класс `Node` описывает внутренний узел (сравнение признака с порогом),
класс `Leaf` — терминальный узел с предсказанием (наиболее частый класс).


In [ ]:
class Node:
    """Внутренний узел: разбиение по признаку index с порогом t."""

    def __init__(self, index, t, true_branch, false_branch):
        self.index = index          # индекс признака, по которому идёт сравнение
        self.t = t                  # значение порога
        self.true_branch = true_branch    # поддерево для условия <= t
        self.false_branch = false_branch  # поддерево для условия > t


class Leaf:
    """Лист: хранит попавшие сюда объекты и предсказанный класс."""

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        # словарь «класс: количество объектов»
        classes = {}
        for label in self.labels:
            classes[label] = classes.get(label, 0) + 1
        # класс с максимальным числом объектов
        return max(classes, key=classes.get)


## 3. Задание 1. Критерий Джини

$$\text{Gini} = 1 - \sum_k p_k^2,$$

где $p_k$ — доля объектов класса $k$ в узле. Для «чистого» узла значение равно 0.


In [ ]:
def gini(labels):
    """Критерий Джини (мера неопределённости узла)."""
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / counts.sum()
    return 1.0 - np.sum(probabilities ** 2)


# Проверка: чистый узел -> 0, равномерный из 2 классов -> 0.5
print('gini чистого узла  :', gini(np.array([0, 0, 0])))
print('gini 2 классов 50/50:', gini(np.array([0, 1])))


## 4. Задание 2. Прирост информации

Прирост = Джини родителя − взвешенная сумма Джини потомков:

$$\text{Gain} = G_{root} - \left(\frac{N_{left}}{N} G_{left} + \frac{N_{right}}{N} G_{right}\right).$$


In [ ]:
def gain(left_labels, right_labels, root_gini):
    """Прирост информации для разбиения узла на две части."""
    total = len(left_labels) + len(right_labels)
    p_left = len(left_labels) / total
    return root_gini - (p_left * gini(left_labels) + (1 - p_left) * gini(right_labels))


## 5. Разбиение данных в узле

Объекты со значением признака `<= t` идут в левое поддерево, остальные — в правое.


In [ ]:
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)

    true_data = data[left]
    false_data = data[right]

    true_labels = labels[left]
    false_labels = labels[right]

    return true_data, false_data, true_labels, false_labels


## 6. Задание 3. Поиск лучшего разбиения и критерии останова

Реализованы **три** критерия останова:

1. **Нет прироста информации** (`gain <= 0`) — узел «чистый» либо разбить нельзя.
2. **Минимальное число объектов в листе** (`min_samples_leaf`) — разбиение,
   при котором хотя бы в одном потомке меньше этого числа, не рассматривается.
3. **Максимальная глубина дерева** (`max_depth`) — по достижении глубины узел
   становится листом.


In [ ]:
MIN_SAMPLES_LEAF = 3    # критерий останова: минимум объектов в листе
MAX_DEPTH = None        # критерий останова: максимум глубины (None — без ограничения)


def find_best_split(data, labels, min_samples_leaf=MIN_SAMPLES_LEAF):
    """Перебирает все признаки и пороги, возвращает наилучшее разбиение."""
    root_gini = gini(labels)

    best_gain = 0.0
    best_t = None
    best_index = None

    n_features = data.shape[1]

    for index in range(n_features):
        t_values = np.unique(data[:, index])

        for t in t_values:
            true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

            # критерий останова: в каждом листе должно быть >= min_samples_leaf объектов
            if len(true_labels) < min_samples_leaf or len(false_labels) < min_samples_leaf:
                continue

            current_gain = gain(true_labels, false_labels, root_gini)

            if current_gain > best_gain:
                best_gain, best_t, best_index = current_gain, t, index

    return best_gain, best_t, best_index


def build_tree(data, labels, depth=0):
    """Рекурсивное построение дерева с критериями останова."""

    # Критерий останова 3: достигнута максимальная глубина
    if MAX_DEPTH is not None and depth >= MAX_DEPTH:
        return Leaf(data, labels)

    gain_value, t, index = find_best_split(data, labels)

    # Критерий останова 1: нет прироста качества (узел чистый / нет допустимого разбиения)
    if t is None or gain_value <= 0:
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

    # Рекурсивно строим два поддерева
    true_branch = build_tree(true_data, true_labels, depth + 1)
    false_branch = build_tree(false_data, false_labels, depth + 1)

    return Node(index, t, true_branch, false_branch)


## 7. Классификация объекта и печать дерева

In [ ]:
def classify_object(obj, node):
    """Спускаемся по дереву от корня до листа и возвращаем предсказание."""
    if isinstance(node, Leaf):
        return node.prediction

    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    else:
        return classify_object(obj, node.false_branch)


def predict(data, tree):
    """Предсказание для набора объектов."""
    return [classify_object(obj, tree) for obj in data]


def print_tree(node, spacing=''):
    """Печать структуры дерева."""
    if isinstance(node, Leaf):
        print(spacing + 'Прогноз:', node.prediction)
        return

    print(spacing + 'Индекс', str(node.index), '<=', str(node.t))

    print(spacing + '--> True:')
    print_tree(node.true_branch, spacing + '  ')

    print(spacing + '--> False:')
    print_tree(node.false_branch, spacing + '  ')


## 8. Задание 4. Метрика качества модели

Точность — доля правильных ответов.


In [ ]:
def accuracy_metric(actual, predicted):
    """Точность: доля совпавших предсказаний."""
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return float(np.mean(actual == predicted))


## 9. Построение самописного дерева и оценка качества

Разбиваем данные на обучающую и тестовую выборки, строим дерево и считаем
точность.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Построение самописного дерева
my_tree = build_tree(X_train, y_train)

print('=== Самописное дерево ===')
print_tree(my_tree)

# Предсказания и точность
my_pred_train = predict(X_train, my_tree)
my_pred_test = predict(X_test, my_tree)

print()
print(f'Точность (train): {accuracy_metric(y_train, my_pred_train):.4f}')
print(f'Точность (test) : {accuracy_metric(y_test, my_pred_test):.4f}')


## 10. Сравнение с деревом решений из sklearn

Строим `DecisionTreeClassifier` с **теми же параметрами** (`criterion='gini'`,
`min_samples_leaf`, `max_depth`) и проверяем, что предсказания совпадают
полностью.


In [ ]:
sklearn_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    splitter='best',
    random_state=42,
)
sklearn_tree.fit(X_train, y_train)

sk_pred_train = sklearn_tree.predict(X_train)
sk_pred_test = sklearn_tree.predict(X_test)

print('=== Сравнение ===')
print(f'Точность sklearn (train): {accuracy_metric(y_train, sk_pred_train):.4f}')
print(f'Точность sklearn (test) : {accuracy_metric(y_test, sk_pred_test):.4f}')

# Проверка полного совпадения предсказаний
train_match = np.array_equal(my_pred_train, sk_pred_train)
test_match = np.array_equal(my_pred_test, sk_pred_test)

print()
print(f'Предсказания совпадают на train: {train_match}')
print(f'Предсказания совпадают на test : {test_match}')


## Вывод

Самописное дерево решений реализует критерий Джини, прирост информации,
три критерия останова (нет прироста, минимальное число объектов в листе,
максимальная глубина) и метрику точности. При одинаковых гиперпараметрах
его предсказания полностью совпадают с предсказаниями `DecisionTreeClassifier`
из `sklearn`.
